In [1]:
from pyspark.sql.functions import *

# --------------------------------------------
# Load tables
# --------------------------------------------

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

dim_date = spark.table("lh_Gold_Telecom.dbo.dim_date")
dim_site = spark.table("lh_Gold_Telecom.dbo.dim_site").filter(col("is_current") == True)
dim_vendor = spark.table("lh_Gold_Telecom.dbo.dim_vendor")
dim_technology = spark.table("lh_Gold_Telecom.dbo.dim_technology")
dim_province = spark.table("lh_Gold_Telecom.dbo.dim_province")
dim_cause = spark.table("lh_Gold_Telecom.dbo.dim_cause")


StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 3, Finished, Available, Finished, False)

In [2]:
# --------------------------------------------
# Create temp views 
# --------------------------------------------

silver_df.createOrReplaceTempView("silver")
dim_date.createOrReplaceTempView("dim_date")
dim_site.createOrReplaceTempView("dim_site")
dim_vendor.createOrReplaceTempView("dim_vendor")
dim_technology.createOrReplaceTempView("dim_technology")
dim_province.createOrReplaceTempView("dim_province")
dim_cause.createOrReplaceTempView("dim_cause")


StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 4, Finished, Available, Finished, False)

In [3]:
%%sql
CREATE OR REPLACE TEMP VIEW fact_src AS
SELECT
    d.date_key,
    s.site_id,
    v.vendor_key,
    t.technology_key,
    p.province_key,
    c.cause_key,

    silver.duration_minutes_corrected,
    silver.availability_percent_corrected,
    silver.is_sla_breach_indicator

FROM silver
JOIN dim_date d
    ON silver.outage_start_date = d.calendar_date
JOIN dim_site s
    ON silver.Site_ID = s.site_id
   AND s.is_current = true
JOIN dim_vendor v
    ON silver.Vendor = v.vendor_name
JOIN dim_technology t
    ON silver.Technology = t.technology_name
JOIN dim_province p
    ON silver.Province_Code = p.province_code
JOIN dim_cause c
    ON silver.Cause = c.cause_name;


StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
SELECT COUNT(*) FROM fact_src;


StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [5]:
%%sql
CREATE OR REPLACE TEMP VIEW fact_outage_by_cause AS
SELECT
    date_key,
    cause_key,
    vendor_key,
    technology_key,
    province_key,
    
    SUM(duration_minutes_corrected) AS total_outage_minutes,
    COUNT(*)                        AS event_count
    
FROM fact_src
GROUP BY
    date_key,
    cause_key,
    vendor_key,
    technology_key,
    province_key;

StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
CREATE OR REPLACE TABLE lh_Gold_Telecom.dbo.fact_outage_cause
USING DELTA
AS
SELECT * FROM fact_outage_by_cause;


StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
# Verify table was created
print("✅ Gold table created successfully!\n")
print(f"fact_outage_cause rows: {spark.table('lh_Gold_Telecom.dbo.fact_outage_cause').count()}")

StatementMeta(, 2af279c0-6ee5-4291-8fd5-acb2d9f96b66, 9, Finished, Available, Finished, False)

✅ Gold tables created successfully!

fact_outage_cause rows: 39826
